# Modernized AlexNet — Scaling CNNs with ReLU, Dropout, and Data Augmentation

## Overview

This notebook implements Modernized AlexNet from first principles in PyTorch. The central architectural idea is aggressive early downsampling, ReLU nonlinearities, dropout regularization, and a large fully connected classifier. The implementation uses Food101 with 101 food categories, and the notebook demonstrates the data pipeline, model construction, inspection, training, evaluation, and prediction workflow.


## Table of Contents

- [Overview](#overview)
- [Learning Objectives](#learning-objectives)
- [Architecture Theory](#architecture-theory)
- [Environment Setup](#environment-setup)
- [Configuration](#configuration)
- [Reproducibility](#reproducibility)
- [Device Selection](#device-selection)
- [Data Augmentation and Preprocessing](#data-augmentation-and-preprocessing)
- [Dataset Construction and Splitting](#dataset-construction-and-splitting)
- [Data Loaders](#data-loaders)
- [Dataset Inspection](#dataset-inspection)
- [Complete Model](#complete-model)
- [Model Initialization](#model-initialization)
- [Model Inspection](#model-inspection)
- [Parameter Count](#parameter-count)
- [Training Objective and Optimizer](#training-objective-and-optimizer)
- [Training Function](#training-function)
- [Evaluation Function](#evaluation-function)
- [Training Loop](#training-loop)
- [Test Evaluation](#test-evaluation)
- [Key Takeaways](#key-takeaways)


## Learning Objectives

By the end of this notebook, we should understand:

- why AlexNet was a decisive scaling step for convolutional networks.
- how large early kernels and stride reduce spatial resolution quickly.
- how data augmentation and dropout regularize a high-capacity classifier.
- why this repository should be read as a modernized AlexNet rather than an exact historical reproduction.
- how the Food101 training split is divided into training and validation subsets.


## Architecture Theory

AlexNet showed that convolutional networks could scale to large natural-image recognition tasks when paired with GPU training, ReLU activations, augmentation, and dropout. The first layers use large kernels and stride to rapidly reduce the spatial grid, while later `3 x 3` convolutions refine increasingly semantic features.

This repository implements a modernized AlexNet for Food101. It preserves the recognizable feature extractor and classifier pattern, but it should not be read as an exact replica of the 2012 system. The model operates on `224 x 224` RGB images and predicts 101 food categories.

| Stage | Operation | Output Shape |
| --- | --- | --- |
| Input | RGB image | `(N, 3, 224, 224)` |
| Conv1 | `11 x 11`, stride 4, `3 -> 64` | `(N, 64, 55, 55)` |
| Pool1 | `3 x 3`, stride 2 | `(N, 64, 27, 27)` |
| Conv2 | `5 x 5`, `64 -> 192` | `(N, 192, 27, 27)` |
| Pool2 | `3 x 3`, stride 2 | `(N, 192, 13, 13)` |
| Conv3-5 | three `3 x 3` convolutions | `(N, 256, 13, 13)` |
| Pool3 | `3 x 3`, stride 2 | `(N, 256, 6, 6)` |
| Classifier | large linear layers with dropout | `(N, 101)` |


## Environment Setup

The notebook uses PyTorch for tensor computation and neural network modules, torchvision for datasets and transforms, NumPy and Python randomness for reproducibility, and Matplotlib for visualization.


In [ ]:
from __future__ import annotations

import random
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms


## Configuration

The following constants define the experiment. They are kept from the source script so the notebook remains traceable to the original implementation.


In [ ]:
DATA_DIRECTORY = "./data"

IMAGE_SIZE = 224
BATCH_SIZE = 32
LEARNING_RATE = 0.01
NUM_EPOCHS = 30

VALIDATION_RATIO = 0.1
RANDOM_SEED = 42
NUMBER_OF_WORKERS = 0

NUMBER_OF_CLASSES = 101


## Reproducibility

Random seeds make dataset splitting, parameter initialization, and stochastic operations easier to reproduce across runs. Exact determinism can still depend on hardware kernels and backend behavior.


In [ ]:
def set_seed(seed: int = RANDOM_SEED) -> None:
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## Device Selection

The model is moved to the best available device. CUDA is preferred when available, followed by Apple MPS, and then CPU.


In [ ]:
def get_device() -> torch.device:
    """Get the available device (GPU or CPU)."""

    if torch.cuda.is_available():
        return torch.device("cuda")

    if torch.backends.mps.is_available():
        return torch.device("mps")

    return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")


## Data Augmentation and Preprocessing

Training transforms are stochastic and regularize the model through random crops or flips. Evaluation transforms are deterministic so validation and test metrics are comparable across runs. Normalization maps image channels into the scale expected by the training recipe.


In [ ]:
# Training transform includes data augmentation.
training_transform = transforms.Compose([
    # Randomly crop part of the image and resize it to 224 x 224.
    transforms.RandomResizedCrop(
        size=IMAGE_SIZE,
        scale=(0.8, 1.0),
    ),

    # Randomly flip the image horizontally.
    transforms.RandomHorizontalFlip(p=0.5),

    # Convert PIL image:
    #
    # (H, W, C)
    #
    # to PyTorch tensor:
    #
    # (C, H, W)
    #
    # and scale pixel values from [0, 255] to [0.0, 1.0].
    transforms.ToTensor(),

    # Normalize the three RGB channels.
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    )
])

# Validation and test transforms must be deterministic.
evaluation_transform = transforms.Compose([
    # Resize the shorter side before taking a center crop.
    transforms.Resize(size=IMAGE_SIZE),

    # Produce the final:
    #
    # (3, 224, 224)
    transforms.CenterCrop(size=IMAGE_SIZE),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5],
    )
])


## Dataset Construction and Splitting

The dataset objects define the supervised image-classification task and its train, validation, and test usage. The source implementation creates validation data from the training split and reserves the official evaluation split for final testing where applicable.


In [ ]:
# Download the training split once.
training_dataset_full = datasets.Food101(
    root=DATA_DIRECTORY,
    split="train",
    download=True,
    transform=training_transform,
)

# Create another view of the same training split,
# but with deterministic validation transforms.
validation_dataset_full = datasets.Food101(
    root=DATA_DIRECTORY,
    split="train",
    download=False,
    transform=evaluation_transform,
)

test_dataset = datasets.Food101(
    root=DATA_DIRECTORY,
    split="test",
    download=False,
    transform=evaluation_transform,
)

# ------------------------------------------------------------
# Generate reproducible train / validation indices.
# ------------------------------------------------------------

number_of_training_samples = len(training_dataset_full)
number_of_validation_samples = int(VALIDATION_RATIO * number_of_training_samples)

generator = torch.Generator().manual_seed(RANDOM_SEED)

indices = torch.randperm(number_of_training_samples, generator=generator).tolist()

validation_indices = indices[:number_of_validation_samples]

training_indices = indices[number_of_validation_samples:]

training_dataset = Subset(training_dataset_full, training_indices)

validation_dataset = Subset(validation_dataset_full, validation_indices)

print(f"Training samples:   {len(training_dataset):,}")
print(f"Validation samples: {len(validation_dataset):,}")
print(f"Test samples:       {len(test_dataset):,}")


## Data Loaders

Data loaders batch examples, optionally shuffle the training subset, and pin memory when CUDA is used. Validation and test loaders are not shuffled because their order does not affect the metrics.


In [ ]:
pin_memory = device.type == "cuda"

training_loader = DataLoader(
    dataset=training_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory,
)

validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUMBER_OF_WORKERS,
    pin_memory=pin_memory,
)


## Dataset Inspection

A small batch visualization helps verify that augmentation, normalization, and labels are wired correctly. The display reverses normalization before plotting.


In [ ]:
def show_training_examples(data_loader: DataLoader, number_of_images: int = 8) -> None:
    images, labels = next(iter(data_loader))
    images = images[:number_of_images]
    labels = labels[:number_of_images]

    mean = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    std = torch.tensor([0.5, 0.5, 0.5]).view(3, 1, 1)
    images = (images.cpu() * std + mean).clamp(0, 1)

    class_names = training_dataset_full.classes
    fig, axes = plt.subplots(1, number_of_images, figsize=(14, 2.5))

    for axis, image, label in zip(axes, images, labels):
        axis.imshow(image.permute(1, 2, 0))
        axis.set_title(class_names[int(label)], fontsize=8)
        axis.axis("off")

    plt.tight_layout()
    plt.show()

show_training_examples(training_loader)


## Complete Model

### Modernized AlexNet Architecture

The model below uses the AlexNet-style sequence of large early kernels, max pooling, later `3 x 3` convolutions, and a dropout-regularized classifier. It is modernized for the repository dataset and PyTorch training loop.


In [ ]:
class AlexNet(nn.Module):
    """Modernized AlexNet image classification network."""
    def __init__(self, number_of_classes: int = NUMBER_OF_CLASSES) -> None:
        super().__init__()

        # ----------------------------------------------------
        # Conv1
        #
        # (N, 3, 224, 224)
        #       ↓
        # (N, 64, 55, 55)
        # ----------------------------------------------------
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=11,
            stride=4,
            padding=2
        )
        self.relu1 = nn.ReLU(inplace=True)

        # ----------------------------------------------------
        # Pool1
        #
        # (N, 64, 55, 55)
        #       ↓
        # (N, 64, 27, 27)
        # ----------------------------------------------------
        self.pool1 = nn.MaxPool2d(
            kernel_size=3,
            stride=2
        )

        # ----------------------------------------------------
        # Conv2
        #
        # (N, 64, 27, 27)
        #       ↓
        # (N, 192, 27, 27)
        # ----------------------------------------------------
        self.conv2 = nn.Conv2d(
            in_channels =64,
            out_channels=192,
            kernel_size=5,
            stride=1,
            padding=2
        )
        self.relu2 = nn.ReLU(inplace=True)

        # ----------------------------------------------------
        # Pool2
        #
        # (N, 192, 27, 27)
        #       ↓
        # (N, 192, 13, 13)
        # ----------------------------------------------------
        self.pool2 = nn.MaxPool2d(
            kernel_size=3,
            stride=2
        )

        # ----------------------------------------------------
        # Conv3
        #
        # (N, 192, 13, 13)
        #       ↓
        # (N, 384, 13, 13)
        # ----------------------------------------------------
        self.conv3 = nn.Conv2d(
            in_channels=192,
            out_channels=384,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.relu3 = nn.ReLU(inplace=True)

        # ----------------------------------------------------
        # Conv4
        #
        # (N, 384, 13, 13)
        #       ↓
        # (N, 256, 13, 13)
        # ----------------------------------------------------
        self.conv4 = nn.Conv2d(
            in_channels=384,
            out_channels=256,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.relu4 = nn.ReLU(inplace=True)

        # ----------------------------------------------------
        # Conv5
        #
        # (N, 256, 13, 13)
        #       ↓
        # (N, 256, 13, 13)
        # ----------------------------------------------------
        self.conv5 = nn.Conv2d(
            in_channels=256,
            out_channels=256,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.relu5 = nn.ReLU(inplace=True)

        # ----------------------------------------------------
        # Pool3
        #
        # (N, 256, 13, 13)
        #       ↓
        # (N, 256, 6, 6)
        # ----------------------------------------------------
        self.pool3 = nn.MaxPool2d(
            kernel_size=3,
            stride=2
        )

        # ----------------------------------------------------
        # Adaptive pooling
        #
        # Guarantee a fixed:
        #
        # (N, 256, 6, 6)
        #
        # before the classifier.
        # ----------------------------------------------------
        self.adaptive_pool = nn.AdaptiveAvgPool2d(
            output_size=(6, 6)
        )

        # ----------------------------------------------------
        # Flatten
        #
        # (N, 256, 6, 6)
        #       ↓
        # (N, 9216)
        #
        # 256 * 6 * 6 = 9216
        # ----------------------------------------------------
        self.flatten = nn.Flatten(start_dim=1)

        # ----------------------------------------------------
        # FC1
        #
        # (N, 9216)
        #       ↓
        # (N, 4096)
        # ----------------------------------------------------

        self.fc1 = nn.Linear(
            in_features=9216,
            out_features=4096
        )
        self.relu6 = nn.ReLU(inplace=True)
        self.dropout1 = nn.Dropout(p=0.5)

        # ----------------------------------------------------
        # FC2
        #
        # (N, 4096)
        #       ↓
        # (N, 4096)
        # ----------------------------------------------------
        self.fc2 = nn.Linear(
            in_features=4096,
            out_features=4096
        )
        self.relu7 = nn.ReLU(inplace=True)
        self.dropout2 = nn.Dropout(p=0.5)

        # ----------------------------------------------------
        # Output layer
        #
        # (N, 4096)
        #       ↓
        # (N, number_of_classes)
        # ----------------------------------------------------
        self.output_layer = nn.Linear(
            in_features=4096,
            out_features=number_of_classes
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward pass.

        Input:
            x: (N, 3, 224, 224)

        Output:
            logits: (N, number_of_classes)
        """

        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.relu3(x)

        x = self.conv4(x)
        x = self.relu4(x)

        x = self.conv5(x)
        x = self.relu5(x)
        x = self.pool3(x)

        x = self.adaptive_pool(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu6(x)
        x = self.dropout1(x)

        x = self.fc2(x)
        x = self.relu7(x)
        x = self.dropout2(x)

        logits = self.output_layer(x)

        return logits


## Model Initialization

The model is instantiated with the configured number of classes and moved to the selected device so parameters and input tensors live on the same backend.


In [ ]:
model = AlexNet(number_of_classes=NUMBER_OF_CLASSES).to(device)

print("Model architecture:")
print(model)


## Model Inspection

Shape inspection is a lightweight sanity check. It verifies that a synthetic input flows through the model and produces logits with the expected class dimension.


In [ ]:
def inspect_model_shapes(
    model: AlexNet,
    device: torch.device,
) -> None:
    """Print tensor shapes after every major model operation."""

    sample_batch = torch.randn(
        4,
        3,
        IMAGE_SIZE,
        IMAGE_SIZE,
        device=device,
    )

    model.eval()

    with torch.inference_mode():
        print("\nShape inspection:")

        print(
            f"Input:          {tuple(sample_batch.shape)}"
        )

        x = model.conv1(sample_batch)

        print(
            f"After Conv1:    {tuple(x.shape)}"
        )

        x = model.relu1(x)
        x = model.pool1(x)

        print(
            f"After Pool1:    {tuple(x.shape)}"
        )

        x = model.conv2(x)

        print(
            f"After Conv2:    {tuple(x.shape)}"
        )

        x = model.relu2(x)
        x = model.pool2(x)

        print(
            f"After Pool2:    {tuple(x.shape)}"
        )

        x = model.conv3(x)

        print(
            f"After Conv3:    {tuple(x.shape)}"
        )

        x = model.relu3(x)

        x = model.conv4(x)

        print(
            f"After Conv4:    {tuple(x.shape)}"
        )

        x = model.relu4(x)

        x = model.conv5(x)

        print(
            f"After Conv5:    {tuple(x.shape)}"
        )

        x = model.relu5(x)
        x = model.pool3(x)

        print(
            f"After Pool3:    {tuple(x.shape)}"
        )

        x = model.adaptive_pool(x)

        print(
            f"After Adaptive: {tuple(x.shape)}"
        )

        x = model.flatten(x)

        print(
            f"After Flatten:  {tuple(x.shape)}"
        )

        x = model.fc1(x)

        print(
            f"After FC1:      {tuple(x.shape)}"
        )

        x = model.relu6(x)
        x = model.dropout1(x)

        x = model.fc2(x)

        print(
            f"After FC2:      {tuple(x.shape)}"
        )

        x = model.relu7(x)
        x = model.dropout2(x)

        logits = model.output_layer(x)

        print(
            f"Logits:         {tuple(logits.shape)}"
        )


inspect_model_shapes(
    model=model,
    device=device,
)


## Parameter Count

Parameter count is a rough measure of model capacity and storage cost. It does not fully measure computation, but it helps compare architectural trade-offs.


In [ ]:
def count_parameters(
    model: nn.Module,
) -> tuple[int, int]:
    """Return total and trainable parameter counts."""

    total_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    return total_parameters, trainable_parameters


total_parameters, trainable_parameters = count_parameters(
    model=model
)


print(
    f"\nTotal parameters:     {total_parameters:,}"
)

print(
    f"Trainable parameters: {trainable_parameters:,}"
)


## Training Objective and Optimizer

For multi-class classification, the network outputs logits. `CrossEntropyLoss` combines log-softmax and negative log-likelihood, so the model should not apply softmax before the loss. The optimizer updates trainable parameters according to the configured learning rate.


In [ ]:
# CrossEntropyLoss expects:
#
# logits:
#   shape = (N, 101)
#
# labels:
#   shape = (N,)
#   dtype = torch.int64
#
# Do not apply Softmax before CrossEntropyLoss.

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=LEARNING_RATE,
)


## Training Function

One training epoch performs forward propagation, loss computation, gradient reset, backpropagation, optimizer update, and metric accumulation.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """Train the model for one epoch."""

    model.train()

    accumulated_loss = 0.0
    number_of_correct_predictions = 0
    number_of_samples = 0

    for images, labels in data_loader:
        images = images.to(device, non_blocking=device.type == "cuda")
        labels = labels.to(device, non_blocking=device.type == "cuda")

        # Remove gradients from the previous iteration.
        optimizer.zero_grad(set_to_none=True)

        # ----------------------------------------------------
        # Forward propagation
        #
        # images:
        #   (N, 3, 224, 224)
        #
        # logits:
        #   (N, 101)
        # ----------------------------------------------------
        logits = model(images)

        # Compute classification loss.
        loss = loss_function(logits, labels)

        # Compute gradients.
        loss.backward()

        # Update trainable parameters.
        optimizer.step()

        current_batch_size = images.size(0)

        accumulated_loss += (
            loss.item()
            * current_batch_size
        )

        number_of_samples += (
            current_batch_size
        )


        predictions = logits.argmax(
            dim=1
        )


        number_of_correct_predictions += (
            predictions == labels
        ).sum().item()


    average_loss = (
        accumulated_loss
        / number_of_samples
    )


    accuracy = (
        number_of_correct_predictions
        / number_of_samples
    )


    return average_loss, accuracy


## Evaluation Function

Evaluation disables gradient tracking and measures loss and accuracy without updating parameters.


In [ ]:
def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate the model without updating parameters."""

    model.eval()

    accumulated_loss = 0.0

    number_of_correct_predictions = 0
    number_of_samples = 0


    with torch.inference_mode():

        for images, labels in data_loader:

            images = images.to(
                device,
                non_blocking=device.type == "cuda",
            )

            labels = labels.to(
                device,
                non_blocking=device.type == "cuda",
            )


            logits = model(images)


            loss = loss_function(
                logits,
                labels,
            )


            current_batch_size = images.size(0)


            accumulated_loss += (
                loss.item()
                * current_batch_size
            )


            number_of_samples += (
                current_batch_size
            )


            predictions = logits.argmax(
                dim=1
            )


            number_of_correct_predictions += (
                predictions == labels
            ).sum().item()


    average_loss = (
        accumulated_loss
        / number_of_samples
    )


    accuracy = (
        number_of_correct_predictions
        / number_of_samples
    )


    return average_loss, accuracy


## Training Loop

The training loop coordinates epochs, validation, and history recording. Running the next cell can take a long time because it executes the full configured experiment.


In [ ]:
training_loss_history: list[float] = []
validation_loss_history: list[float] = []

training_accuracy_history: list[float] = []
validation_accuracy_history: list[float] = []


for epoch in range(1, NUM_EPOCHS + 1):

    training_loss, training_accuracy = train_one_epoch(
        model=model,
        data_loader=training_loader,
        loss_function=loss_function,
        optimizer=optimizer,
        device=device,
    )


    validation_loss, validation_accuracy = evaluate(
        model=model,
        data_loader=validation_loader,
        loss_function=loss_function,
        device=device,
    )


    training_loss_history.append(
        training_loss
    )

    validation_loss_history.append(
        validation_loss
    )


    training_accuracy_history.append(
        training_accuracy
    )

    validation_accuracy_history.append(
        validation_accuracy
    )


    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] | "
        f"Training loss: {training_loss:.4f} | "
        f"Training accuracy: {training_accuracy:.2%} | "
        f"Validation loss: {validation_loss:.4f} | "
        f"Validation accuracy: {validation_accuracy:.2%}"
    )


## Test Evaluation

The test set is used after training for final evaluation. Training data optimizes parameters, validation data monitors model selection, and test data estimates final generalization.


In [ ]:
test_loss, test_accuracy = evaluate(
    model=model,
    data_loader=test_loader,
    loss_function=loss_function,
    device=device,
)


print("\nTest results:")

print(
    f"Test loss:     {test_loss:.4f}"
)

print(
    f"Test accuracy: {test_accuracy:.2%}"
)


## Key Takeaways

- AlexNet scales CNN capacity through a deeper feature extractor and a large classifier.
- The implementation uses augmentation and dropout to reduce overfitting on Food101.
- This is a modernized AlexNet-style implementation, not a bit-for-bit historical reproduction.
